# CLSTM trajectory prediction: trained on the clean data and on the 2stage-CPLS output

This notebook trains the **Context-aware LSTM (CLSTM)** of

> Mehri, S., Alesheikh, A. A., Basiri, A. (2021). *A Contextual Hybrid Model for Vessel Movement Prediction.* IEEE Access 9, 45600-45613. [doi:10.1109/ACCESS.2021.3066463](https://doi.org/10.1109/ACCESS.2021.3066463)

to predict where a vessel will be over its next three time steps, and it does so **twice**:

* once on the **clean, uncompressed** AIS trips, and
* once on the same trips **compressed with 2stage-CPLS** (notebook 03),

then compares the two. For each of the two datasets it fits three predictors on the same footing:

* the **curvilinear model** (dead reckoning from speed and course, no learning): the baseline;
* the **ordinary LSTM**: one network for all vessels, `ClstmPredictor(context=False)`;
* the **CLSTM**: one network per vessel type, `ClstmPredictor(context=True)`. The vessel type is the *context*.

Sections:

1. **Inputs**: the trips and their 2stage-CPLS compression written by notebook 03
2. **Samples**: Eqs. 21 and 22 of the paper, built for both datasets
3. **One split for both datasets**: the same trips are held out for testing in both
4. **Training**: four fits (LSTM and CLSTM, on each dataset), on the GPU when there is one
5. **Evaluation**: errors per vessel type, CLSTM against LSTM, and each model on the other dataset's test samples
6. **An example**, then **save**
7. **Notes**: what the comparison means and what it does not

All charts are interactive (hvplot/Holoviews, Bokeh): zoom, pan and hover. GitHub's notebook viewer does not run JavaScript, so they do not show on github.com; open the notebook in Jupyter or on [nbviewer](https://nbviewer.org/github/smehri/CATraj/blob/main/notebooks/05_clstm_prediction.ipynb).

**Inputs**: `results/03_two_stage_cpls_compression/<DATA>/trips_ais.parquet` and `compressed_ais.parquet`.
**Outputs**: `results/05_clstm_prediction/<DATA>/` with `errors.csv`, `summary.txt` and the four trained models.

Run this notebook live (interactive figures) on Binder: [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/smehri/CATraj/main?labpath=notebooks%2F05_clstm_prediction.ipynb) · [nbviewer](https://nbviewer.org/github/smehri/CATraj/blob/main/notebooks/05_clstm_prediction.ipynb). github.com does not run the JavaScript of the interactive figures.

## Data

The notebook reads two files that `03_two_stage_cpls_compression.ipynb` wrote for the chosen dataset:
`trips_ais.parquet` (the clean, median-filtered AIS points split into trips at gaps) and
`compressed_ais.parquet` (the 2stage-CPLS output of exactly those trips). The AIS is from the
Brest area of the NARI dataset (Ray et al. 2019, [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141);
Zenodo [doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595); CC BY-NC-SA 4.0, so everything derived
from it, this notebook's outputs included, is shared under the same licence).

By default the notebook runs on the **committed, preprocessed two-day sample** (2015-11-09 to 2015-11-11 UTC,
Brest area, 565 vessels). **The sample is far too small for meaningful accuracy**: it holds a few hundred
trips and, for most vessel types, a handful of vessels, so the numbers below only show that the code runs and
what the outputs look like. To process the **full archive**, run `notebooks/01_preprocessing.ipynb` first (its download
and preprocessing sections), then `notebooks/03_two_stage_cpls_compression.ipynb` with the same setting, and then set
`DATA = "full"` below, or run with the environment variable `CATRAJ_DATA=full`. The full archive covers
October to December 2015 and February to March 2016. **January 2016 is left out of every dataset**: the Zenodo archive has no
sea state for it (its `oc_january.csv` is a byte copy of `oc_november.csv`), so those positions could not be annotated with the
ocean context and are not used anywhere.

In [1]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")   # see catraj/__init__.py

DATA = os.environ.get("CATRAJ_DATA", "sample")      # "sample" or "full"
N_JOBS = int(os.environ.get("CATRAJ_N_JOBS", "-1"))  # -1: every core; 1: serial
DEVICE = os.environ.get("CATRAJ_DEVICE", "auto")     # "auto": the GPU when there is one; "cpu" to force the CPU

import time
from pathlib import Path

import holoviews as hv
import hvplot.pandas  # noqa: F401  (adds .hvplot to DataFrames)
from pyproj import Transformer
import numpy as np
import pandas as pd
import geopandas as gpd
import torch

from catraj.datasets import repo_root
from catraj.prediction import (ClstmPredictor, make_samples, prediction_errors,
                               split_by_trajectory)

if DATA not in ("sample", "full"):
    raise ValueError(f"DATA must be 'sample' or 'full', not {DATA!r}")
ROOT = repo_root()
IN = ROOT / "results" / "03_two_stage_cpls_compression" / DATA
OUT = ROOT / "results" / "05_clstm_prediction" / DATA
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "progress.log").unlink(missing_ok=True)   # a plain-text log of each fit, handy to follow a long run

# ---- the method's settings: the library defaults (H1-H6 of the CLSTM brief) are used as they are ----
TEST_SIZE, SEED = 0.2, 0
CRS = "EPSG:32630"
BLUE, ORANGE, GREEN, GREY = "#1f77b4", "#ff7f0e", "#2ca02c", "#7f7f7f"
TO_MERCATOR = Transformer.from_crs(32630, 3857, always_xy=True).transform   # for the basemap

hv.extension("bokeh")

resolved = "cuda" if (DEVICE == "auto" and torch.cuda.is_available()) else ("cpu" if DEVICE == "auto" else DEVICE)
GPU_NAME = torch.cuda.get_device_name(0) if resolved.startswith("cuda") else "none (CPU)"
print(f"DATA = {DATA!r}   device = {resolved!r}   GPU = {GPU_NAME}   torch {torch.__version__}")

DATA = 'full'   device = 'cuda'   GPU = NVIDIA GeForce RTX 4060 Ti   torch 2.11.0+cu128


### Parallel processing

* **Training runs on the GPU when there is one.** `ClstmPredictor(device="auto")` uses CUDA if PyTorch finds it and
  the CPU otherwise. The networks are tiny (one LSTM layer of 64 units over three steps), so a GPU is mostly a way of
  getting through the millions of mini-batches of the full archive; the code is identical on both. The training of the
  networks is **not** spread over processes: the groups (vessel types) are trained one after the other, each from the same seed,
  so that one group's network never depends on another's.
* **Building the samples** is vectorised inside each trip (numpy), and the trips are independent. Here the trips are
  processed one after the other in this process: `make_samples` takes no `n_jobs` because it is not the slow
  step (the timings are printed below). `N_JOBS` is therefore not used by this notebook; it is set only so that all
  notebooks share one settings cell. Nothing in this notebook depends on the number of cores, so a run is the same everywhere.
* **Memory on the full archive.** The 14.8 million clean points give about as many samples
  (`inputs` of shape `(m, 3, 3)` and `targets` of shape `(m, 3, 2)` in float64, 9 + 6 numbers per sample, plus keys);
  the whole of a network's training samples is also copied to the device in float32. Keep roughly 10 GB of free RAM for the full run.
* **CPU only**: set `CATRAJ_DEVICE=cpu` to force the CPU (the same code, slower: see the timing printed under *Training*).
  On the CPU two fits with the same seed give identical predictions; on a GPU the results can differ in the last digits between runs.

## 1 · Inputs

Both files carry the same columns, and both are indexed by the same `trip_id`: a trip is one stretch of one vessel's track
between gaps (notebook 03). The clean file holds every point of the trip; the compressed file holds the points that 2stage-CPLS
kept, plus the `stay` flag of its stay points. **A trajectory is a trip** (C14 of the CLSTM brief): `make_samples` is called with
`by="trip_id"`, so a trip is never mixed with another and the time step of Eq. 12 is computed per trip.

If a file is missing the cell stops and says so.

In [2]:
TRIPS_FILE, COMPRESSED_FILE = IN / "trips_ais.parquet", IN / "compressed_ais.parquet"
missing = [f.name for f in (TRIPS_FILE, COMPRESSED_FILE) if not f.is_file()]
if missing:
    raise FileNotFoundError(
        f"{missing} not found in {IN}. Run notebooks/03_two_stage_cpls_compression.ipynb first "
        f"with DATA = {DATA!r} (CATRAJ_DATA={DATA}); it writes the inputs of this notebook.")

COLUMNS = ["trip_id", "mmsi", "unix_time", "x", "y", "speed_over_ground", "course_over_ground", "vessel_type"]
print("trips_ais.parquet columns:     ", list(pd.read_parquet(TRIPS_FILE).head(0).columns))
print("compressed_ais.parquet columns:", list(pd.read_parquet(COMPRESSED_FILE).head(0).columns))
raw = pd.read_parquet(TRIPS_FILE, columns=COLUMNS)
cpls = pd.read_parquet(COMPRESSED_FILE, columns=COLUMNS)

# "Unknown" is how the preprocessing names a vessel with no usable static type. The paper's group "Other"
# holds the vessels whose type is null, so those two are one group here: map Unknown to Other.
for df in (raw, cpls):
    df["vessel_type"] = df["vessel_type"].replace("Unknown", "Other")

summary_in = pd.DataFrame({
    "clean (uncompressed)": [len(raw), raw.trip_id.nunique(), raw.mmsi.nunique()],
    "2stage-CPLS": [len(cpls), cpls.trip_id.nunique(), cpls.mmsi.nunique()]},
    index=["points", "trips", "vessels"])
summary_in["2stage-CPLS"] = summary_in["2stage-CPLS"].astype(int)
print()
print(summary_in.to_string())
print(f"\ncompression ratio: {100 * (1 - len(cpls) / len(raw)):.2f} % of the points removed")
print(f"mean points per trip: {len(raw) / raw.trip_id.nunique():.1f} clean, {len(cpls) / cpls.trip_id.nunique():.1f} compressed")
types = pd.DataFrame({"clean points": raw.groupby("vessel_type").size(),
                      "compressed points": cpls.groupby("vessel_type").size(),
                      "trips": raw.groupby("vessel_type").trip_id.nunique()}).sort_values("clean points", ascending=False)
types

trips_ais.parquet columns:      ['mmsi', 'trip_id', 'unix_time', 'x', 'y', 'lon', 'lat', 'speed_over_ground', 'course_over_ground', 'vessel_type', 'navigational_status', 'true_heading', 'rate_of_turn', 'ship_type_code']
compressed_ais.parquet columns: ['mmsi', 'trip_id', 'unix_time', 'x', 'y', 'lon', 'lat', 'speed_over_ground', 'course_over_ground', 'vessel_type', 'navigational_status', 'true_heading', 'rate_of_turn', 'ship_type_code', 'stay']



         clean (uncompressed)  2stage-CPLS
points               14777418      1636505
trips                   19407        19407
vessels                  2076         2076

compression ratio: 88.93 % of the points removed
mean points per trip: 761.4 clean, 84.3 compressed


,clean points,compressed points,trips
vessel_type,,,
Passenger,4045030,442929,1311
Fishing,2866090,445859,7339
Other,2784130,245544,2466
Cargo,1252446,239527,4082
Tug,1213339,15824,156
Tanker,626062,69512,1038
Military Ops,540227,20158,103
Pilot Vessel,308202,66132,445
Sailing Vessel,284513,13756,1113


## 2 · Samples

A sample (Eqs. 21 and 22 of the paper) is built from one trip:

* **Input**: the three positions `(x', y')` that the curvilinear model predicts from a vertex's speed over ground (knots, converted
  to m/s) and course over ground, at the offsets `tau_j = j * dt`, `j = 1, 2, 3`. The input of step `j` is `(x'_j, y'_j, tau_j)`.
* **Target**: where the trip actually is at those three times, interpolated linearly in time along the same trip.
* **Time step `dt`** of a trip is Eq. 12: `(t_last - t_first) / (n - 1)`, the mean interval between its vertices.
* **Every vertex is a prediction origin**; a vertex whose speed or course is not available (SOG >= 102.3 knots, COG >= 360, or missing)
  gives no sample, and neither does one whose `t + 3 dt` lies after the end of the trip.

The same function builds both sets. The two sets differ, and that matters later: on the clean trips `dt` is the trip's raw mean
sampling interval (tens of seconds); on the compressed trip it is the mean interval between the *kept* vertices, many times larger,
so a compressed sample predicts a much longer horizon, and its origins are the kept vertices only.

In [3]:
def as_points(df):
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y), crs=CRS)

def build(df, label):
    t0 = time.perf_counter()
    samples = make_samples(as_points(df), speed_column="speed_over_ground", course_column="course_over_ground",
                           context_column="vessel_type", time_column="unix_time", by="trip_id")
    seconds = time.perf_counter() - t0
    print(f"{label}: {len(df):,} points in {df.trip_id.nunique():,} trips -> {len(samples):,} samples "
          f"from {len(set(samples.trajectory.tolist())):,} trips in {seconds:.1f} s")
    return samples, seconds

samples_raw, SAMPLES_RAW_S = build(raw, "clean       ")
samples_cpls, SAMPLES_CPLS_S = build(cpls, "2stage-CPLS ")
dt_raw = pd.Series(samples_raw.dt).groupby(pd.Series(samples_raw.trajectory)).first()
dt_cpls = pd.Series(samples_cpls.dt).groupby(pd.Series(samples_cpls.trajectory)).first()
print(f"\nmedian dt per trip (Eq. 12): {dt_raw.median():.0f} s clean, {dt_cpls.median():.0f} s compressed")
print(f"trips with no sample: {raw.trip_id.nunique() - len(dt_raw):,} clean, {cpls.trip_id.nunique() - len(dt_cpls):,} compressed")

clean       : 14,777,418 points in 19,407 trips -> 14,167,226 samples from 19,337 trips in 32.8 s


2stage-CPLS : 1,636,505 points in 19,407 trips -> 1,559,338 samples from 15,094 trips in 17.8 s



median dt per trip (Eq. 12): 50 s clean, 125 s compressed
trips with no sample: 70 clean, 4,313 compressed


## 3 · One split for both datasets

The data are split **by trip**, never by sample (C13): the samples of a held-out trip are never seen during training, and neighbouring
samples of one trip, which are almost the same, cannot leak across the split. The split follows
`split_by_trajectory(test_size=0.2, seed=0)`: the trip ids, in order of first appearance, are shuffled with `numpy.random.default_rng(0)`
and the first 20 % are the test trips.

It is made **once**, on the clean samples, and the **same train and test trip ids are applied to the compressed samples**. The two
trainings are therefore evaluated on the same held-out trips, which is what makes their results comparable at all. (Splitting each set
separately would give different test trips, because the set of trips that yield samples is not the same.)

In [4]:
train_raw, test_raw = split_by_trajectory(samples_raw, TEST_SIZE, SEED)
test_ids = set(test_raw.trajectory.tolist())
raw_ids = set(samples_raw.trajectory.tolist())
unseen = set(samples_cpls.trajectory.tolist()) - raw_ids
assert not unseen, f"{len(unseen)} compressed trips have no clean sample; cannot place them in the split"
in_test = np.fromiter((i in test_ids for i in samples_cpls.trajectory.tolist()), bool, len(samples_cpls))
train_cpls, test_cpls = samples_cpls.subset(~in_test), samples_cpls.subset(in_test)

train_ids = set(train_raw.trajectory.tolist())
assert train_ids.isdisjoint(test_ids)
assert set(train_cpls.trajectory.tolist()) <= train_ids and set(test_cpls.trajectory.tolist()) <= test_ids
print(f"trips: {len(train_ids):,} train, {len(test_ids):,} test (test share {len(test_ids) / len(raw_ids):.1%})")
split_table = pd.DataFrame({
    "clean: train": pd.Series(train_raw.group).value_counts(), "clean: test": pd.Series(test_raw.group).value_counts(),
    "2stage-CPLS: train": pd.Series(train_cpls.group).value_counts(), "2stage-CPLS: test": pd.Series(test_cpls.group).value_counts(),
}).fillna(0).astype(int).sort_values("clean: train", ascending=False)
split_table.loc["all"] = split_table.sum()
split_table

trips: 15,470 train, 3,867 test (test share 20.0%)


,clean: train,clean: test,2stage-CPLS: train,2stage-CPLS: test
Passenger,3281072,758711,348817,88165
Fishing,2301588,535484,343435,79860
Other,1864674,460005,193122,39733
Tug,1009727,203091,11941,2644
Cargo,970731,249350,179433,40892
Tanker,546874,71697,55411,9440
Military Ops,405205,85508,15668,3616
Pilot Vessel,255574,50885,53896,10190
Pleasure Craft,181312,19684,2212,663
Sailing Vessel,168850,102997,6635,3815


## 4 · Training

Each fit uses the **library defaults**, which are the paper's design choices and are not tuned here: an LSTM layer of 64 units over the three
steps followed by a linear layer to the two coordinates (H1); inputs and learned targets (the deviation, see the section above) standardised with the training statistics (H2); mean squared error (H3);
Adam with learning rate 1e-3, batch size 256, at most 200 epochs (H4); early stopping on 10 % of the training trips, stopping after
10 epochs without improvement and restoring the best weights (H5); seed 0 (H6). A vessel-type group with fewer than 200 training samples
gets no network (`dropped_groups_`) and its predictions are NaN, so it does not appear in the scored samples.

The ordinary LSTM is the same network trained once on every vessel type together; the CLSTM trains one network per type. Four fits follow:
LSTM and CLSTM on the clean samples, then on the compressed samples.

In [5]:
models, fit_seconds = {}, {}
plan = [("lstm_uncompressed", False, train_raw), ("clstm_uncompressed", True, train_raw),
        ("lstm_2stage_cpls", False, train_cpls), ("clstm_2stage_cpls", True, train_cpls)]
for name, context, train in plan:
    t0 = time.perf_counter()
    model = ClstmPredictor(context=context, device=DEVICE).fit(train)
    if resolved.startswith("cuda"):
        torch.cuda.synchronize()
    fit_seconds[name] = time.perf_counter() - t0
    models[name] = model
    epochs = sum(len(h) for h in model.history_.values())
    print(f"{name:20s} {len(train):>10,} samples  {len(model.groups_):2d} network(s)  "
          f"{epochs:5d} epochs in total  {fit_seconds[name]:8.1f} s", flush=True)
    with open(OUT / "progress.log", "a") as log:
        log.write(f"{time.strftime('%H:%M:%S')} {name}: {len(train)} samples, {epochs} epochs, {fit_seconds[name]:.1f} s" + chr(10))
print(f"\ndevice: {models['lstm_uncompressed'].device}   GPU: {GPU_NAME}   total training time: {sum(fit_seconds.values()):.1f} s")

lstm_uncompressed    11,476,290 samples   1 network(s)     17 epochs in total     825.9 s


clstm_uncompressed   11,476,290 samples  17 network(s)    709 epochs in total    1682.6 s


lstm_2stage_cpls      1,264,398 samples   1 network(s)     32 epochs in total     167.9 s


clstm_2stage_cpls     1,264,398 samples  17 network(s)    804 epochs in total     212.5 s



device: cuda   GPU: NVIDIA GeForce RTX 4060 Ti   total training time: 2888.9 s


`summary()` lists, per network, the number of training samples, the epochs run and the best validation loss (on the standardised outputs, so
the values of different networks are comparable); the last line names the groups that were too small to get a network.

In [6]:
for name, model in models.items():
    print(f"== {name} ==")
    print(model.summary())
    print(f"   groups dropped: {model.dropped_groups_}\n")

== lstm_uncompressed ==
LSTM, one network; learns the deviation; device cuda
  all: 11476290 samples, 17 epochs, best validation loss 9.42823
dropped (fewer than 200 samples): none
   groups dropped: []

== clstm_uncompressed ==
CLSTM, one network per group; learns the deviation; device cuda
  Anti-Pollution: 91967 samples, 17 epochs, best validation loss 1.19700
  Cargo: 970731 samples, 11 epochs, best validation loss 0.07927
  Dive Vessel: 9892 samples, 82 epochs, best validation loss 0.85946
  Dredger: 139751 samples, 12 epochs, best validation loss 0.00639
  Fishing: 2301588 samples, 56 epochs, best validation loss 0.74893
  High-Speed Craft: 7075 samples, 200 epochs, best validation loss none
  Law Enforce: 28126 samples, 11 epochs, best validation loss 5.23113
  Local Vessel: 71376 samples, 11 epochs, best validation loss 0.60351
  Military Ops: 405205 samples, 11 epochs, best validation loss 1.31156
  Other: 1864674 samples, 13 epochs, best validation loss 0.27338
  Passenger: 3

**Training curves.** The training loss (mean over the epoch's mini-batches) and the validation loss (held-out trips) per epoch. The validation loss
stops improving, and training stops, 10 epochs later; the weights of the best epoch are the ones kept. Top: the ordinary LSTM. Bottom: the validation
loss of each CLSTM network (click a legend entry to hide a curve). A small network trained on few samples stops early.

In [7]:
def curves(model):
    rows = [(g, e, tr, va) for g, h in sorted(model.history_.items()) for e, tr, va in h]
    return pd.DataFrame(rows, columns=["group", "epoch", "training", "validation"])

panels = []
for tag, title in [("uncompressed", "clean"), ("2stage_cpls", "2stage-CPLS")]:
    d = curves(models[f"lstm_{tag}"]).melt(["group", "epoch"], ["training", "validation"], var_name="loss", value_name="MSE")
    panels.append(d.hvplot.line(x="epoch", y="MSE", by="loss", logy=True, width=520, height=330, color=[BLUE, ORANGE],
                                title=f"LSTM on {title} samples (standardised MSE)"))
for tag, title in [("uncompressed", "clean"), ("2stage_cpls", "2stage-CPLS")]:
    d = curves(models[f"clstm_{tag}"]).dropna(subset=["validation"])
    panels.append(d.hvplot.line(x="epoch", y="validation", by="group", logy=True, width=520, height=330,
                                title=f"CLSTM on {title} samples: validation loss per vessel type"))
hv.Layout(panels).cols(2)

:Layout
   .NdOverlay.I   :NdOverlay   [loss]
      :Curve   [epoch]   (MSE)
   .NdOverlay.II  :NdOverlay   [loss]
      :Curve   [epoch]   (MSE)
   .NdOverlay.III :NdOverlay   [group]
      :Curve   [epoch]   (validation)
   .NdOverlay.IV  :NdOverlay   [group]
      :Curve   [epoch]   (validation)

## What the network learns: the deviation, not the position

The paper's network corrects the curvilinear prediction: the deviation of Eq. 2 is `delta_k = (x_k - x'_k) i + (y_k - y'_k) j`, the difference between the actual position and the
curvilinear one. The library's `ClstmPredictor` therefore has `target="deviation"` (the default, used throughout this notebook): the network is trained to output `delta` for the three steps, and
`predict` returns `T' + delta_hat`, so **the final output is still an absolute position** and the errors below are errors of absolute positions, comparable with the curvilinear baseline.
`target="absolute"` is the first formulation (the network regresses the absolute UTM coordinates themselves).

The change was made with the author on 2026-11-25 after a diagnostic on the full archive (clean trips, vessel type Cargo, 1.03 million training samples, 30 epochs, library defaults, same split;
mean and median point-wise error on held-out trips; run outside this notebook):

| what the network learns | mean error | median error |
|---|---|---|
| nothing (curvilinear model alone) | 59.2 m | 15.6 m |
| absolute positions (`target="absolute"`) | 95.5 m | 52.2 m |
| deviation `delta` (`target="deviation"`) | 60.8 m | 18.4 m |
| positions relative to the origin (local coordinates; not in the library) | 57.7 m | 13.9 m |

The reason is resolution. Absolute UTM coordinates, standardised over the whole area, have a standard deviation of tens of kilometres; a standardised error `e` is `e * std` metres, so even a
standardised RMSE of about 1.7e-3 is about 70 m, more than the curvilinear error over a horizon of seconds. The deviation is small and centred, so its standard deviation is of the order of the
curvilinear error itself and the network works at the right scale. The cell below prints both spreads for this run. In this formulation the CLSTM can only improve on dead reckoning by the amount the
vessel-type context explains of the deviation; if it explains little, networks end up close to the curvilinear baseline.

In [8]:
for dataset, train in [("uncompressed", train_raw), ("2stage_cpls", train_cpls)]:
    absolute = train.targets.reshape(-1, 2).std(axis=0)
    deviation = (train.targets - train.curvilinear).reshape(-1, 2)
    print(f"{dataset:13s} std of the absolute positions: x {absolute[0]:>9,.0f} m, y {absolute[1]:>9,.0f} m   "
          f"std of the deviation: x {deviation[:, 0].std():>8,.1f} m, y {deviation[:, 1].std():>8,.1f} m   "
          f"median |deviation| {np.median(np.hypot(*deviation.T)):.1f} m")

uncompressed  std of the absolute positions: x    19,686 m, y    19,593 m   std of the deviation: x    468.4 m, y    108.1 m   median |deviation| 2.3 m
2stage_cpls   std of the absolute positions: x    32,371 m, y    28,920 m   std of the deviation: x  3,835.4 m, y  3,043.0 m   median |deviation| 65.8 m


## 5 · Evaluation

Errors are on the **held-out test trips**. `prediction_errors` gives, per vessel type and overall, the number of scored samples, the root-mean-square
error (Eq. 25, over the three steps and the two coordinates), the RMSE of each step, and the mean and median **point-wise error**: the distance in metres between
the predicted and the actual position. The mean point-wise error is the measure in which the paper reports a 15.68 % improvement of CLSTM over the LSTM, on
its own data, period and vessel types. **No reproduction of that figure is claimed here**: the data, the period and the types are different, and the comparison
below is an independent experiment.

Two things to keep in mind while reading the tables:

* A vessel type with no network (too few training samples) has NaN predictions in the CLSTM, so the CLSTM's `all` row scores fewer samples than the LSTM's. For a like-for-like
  improvement the LSTM is re-scored on exactly the samples the CLSTM predicted (`LSTM, same samples`).
* The clean and the compressed test sets have different samples and a different horizon (`dt`), so their errors are **not** comparable with each other in absolute terms; see section 7.

In [9]:
def scored(model, samples):
    pred = model.predict(samples)
    return pred, prediction_errors(pred, samples)

TEST = {"uncompressed": test_raw, "2stage_cpls": test_cpls}
preds, tables = {}, {}
errors = []
for dataset in ("uncompressed", "2stage_cpls"):
    test = TEST[dataset]
    t0 = time.perf_counter()
    tables[dataset, "curvilinear"] = prediction_errors(test.curvilinear, test)
    for kind in ("lstm", "clstm"):
        preds[dataset, kind], tables[dataset, kind] = scored(models[f"{kind}_{dataset}"], test)
    print(f"{dataset}: predictions and errors on {len(test):,} test samples in {time.perf_counter() - t0:.1f} s")
    # like for like: the LSTM on the samples the CLSTM predicts (a group without a network is NaN in the CLSTM)
    ok = ~np.isnan(preds[dataset, "clstm"]).any(axis=(1, 2))
    sub = test.subset(ok)
    tables[dataset, "curvilinear_same"] = prediction_errors(sub.curvilinear, sub)
    tables[dataset, "lstm_same"] = prediction_errors(preds[dataset, "lstm"][ok], sub)
    tables[dataset, "clstm_same"] = prediction_errors(preds[dataset, "clstm"][ok], sub)

uncompressed: predictions and errors on 2,690,936 test samples in 7.4 s


2stage_cpls: predictions and errors on 294,940 test samples in 1.2 s


### Clean (uncompressed) data

`curvilinear` is the physical model alone (the baseline both networks start from), `LSTM` the single network, `CLSTM` one network per type. Lengths are in metres.

In [10]:
def show(dataset, kinds=("curvilinear", "lstm", "clstm")):
    label = {"curvilinear": "curvilinear", "lstm": "LSTM", "clstm": "CLSTM"}
    for kind in kinds:
        print(f"--- {label[kind]} ---")
        print(tables[dataset, kind].round(2).to_string())
        print()

show("uncompressed")

--- curvilinear ---
                  samples  missing    rmse  rmse_t+1  rmse_t+2  rmse_t+3  mean_error  median_error
Anti-Pollution      21889        0  164.74     49.60    141.03    243.05       33.83          5.73
Cargo              249350        0   49.58     21.86     41.89     71.70       19.30          7.36
Dive Vessel           525        0  384.69    153.81    346.59    547.88      119.92         35.72
Dredger             31854        0   99.63     33.62     84.74    146.51       32.06         15.75
Fishing            535484        0   88.10     35.49     77.91    126.32       25.57          6.77
High-Speed Craft        8        0  126.79     63.87    115.75    175.36      113.99        108.07
Law Enforce          3043        0  127.09     79.47    126.46    161.70       14.87          6.97
Local Vessel        27341        0    5.80      3.98      5.83      7.14        3.22          1.63
Military Ops        85508        0   23.98      9.35     20.58     34.85        5.08     

In [11]:
def improvement(dataset, tabs=None):
    tabs = tables if tabs is None else tabs
    # relative improvement of the CLSTM over the LSTM in mean point-wise error, on the samples both predict
    lstm, clstm, curv = (tabs[dataset, k + "_same"]["mean_error"] for k in ("lstm", "clstm", "curvilinear"))
    out = pd.DataFrame({"samples": tabs[dataset, "clstm_same"]["samples"], "curvilinear (m)": curv,
                        "LSTM (m)": lstm, "CLSTM (m)": clstm,
                        "CLSTM vs LSTM (%)": 100 * (lstm - clstm) / lstm,
                        "LSTM vs curvilinear (%)": 100 * (curv - lstm) / curv,
                        "CLSTM vs curvilinear (%)": 100 * (curv - clstm) / curv})
    return out

imp = {d: improvement(d) for d in ("uncompressed", "2stage_cpls")}
imp["uncompressed"].round(2)

,samples,curvilinear (m),LSTM (m),CLSTM (m),CLSTM vs LSTM (%),LSTM vs curvilinear (%),CLSTM vs curvilinear (%)
Anti-Pollution,21889,33.83,93.98,43.41,53.80,-177.81,-28.34
Cargo,249350,19.30,27.94,43.71,-56.45,-44.75,-126.46
Dive Vessel,525,119.92,139.58,138.96,0.45,-16.40,-15.88
Dredger,31854,32.06,42.54,35.89,15.64,-32.72,-11.96
Fishing,535484,25.57,31.72,26.86,15.32,-24.07,-5.06
High-Speed Craft,8,113.99,170.05,112.02,34.13,-49.17,1.73
Law Enforce,3043,14.87,20.16,14.86,26.28,-35.64,0.01
Local Vessel,27341,3.22,8.76,3.30,62.36,-171.59,-2.22
Military Ops,85508,5.08,10.28,5.78,43.78,-102.43,-13.80
Other,460005,15.76,24.83,17.51,29.48,-57.52,-11.08


The table above is the paper's measure (mean point-wise error, relative improvement `100 * (LSTM - CLSTM) / LSTM`), per type and for all scored samples (`all`).

### 2stage-CPLS data

In [12]:
show("2stage_cpls")
imp["2stage_cpls"].round(2)

--- curvilinear ---
                  samples  missing      rmse  rmse_t+1  rmse_t+2  rmse_t+3  mean_error  median_error
Anti-Pollution        803        0  20486.06   8093.63  18278.61  29315.90     7960.86       1404.60
Cargo               40892        0   2770.50   1148.91   2453.24   3960.88      404.83         48.77
Dive Vessel            88        0   2218.58    805.97   1969.24   3199.80     1456.91        825.51
Dredger              3341        0   3631.36   1677.71   3334.05   5062.57      883.27         28.65
Fishing             79860        0    856.15    341.32    763.91   1224.31      341.59         64.48
High-Speed Craft        3        0    337.23    158.84    312.13    467.46      306.01        276.11
Law Enforce          1205        0    231.78    261.41    214.83    216.05       35.07         15.95
Local Vessel          652        0   1298.08    514.92   1154.23   1859.46      763.05        400.87
Military Ops         3616        0   5147.68   2036.29   4762.34   7257

,samples,curvilinear (m),LSTM (m),CLSTM (m),CLSTM vs LSTM (%),LSTM vs curvilinear (%),CLSTM vs curvilinear (%)
Anti-Pollution,803,7960.86,2179.04,4222.48,-93.78,72.63,46.96
Cargo,40892,404.83,420.06,450.51,-7.25,-3.76,-11.28
Dive Vessel,88,1456.91,1209.98,1082.74,10.52,16.95,25.68
Dredger,3341,883.27,515.90,1397.61,-170.91,41.59,-58.23
Fishing,79860,341.59,368.80,300.67,18.48,-7.97,11.98
High-Speed Craft,3,306.01,531.43,233.07,56.14,-73.66,23.84
Law Enforce,1205,35.07,175.89,39.65,77.46,-401.54,-13.05
Local Vessel,652,763.05,736.55,844.24,-14.62,3.47,-10.64
Military Ops,3616,1900.14,1466.67,1677.81,-14.40,22.81,11.70
Other,39733,1242.68,758.39,885.48,-16.76,38.97,28.74


In [13]:
bars = []
for dataset, title in [("uncompressed", "clean"), ("2stage_cpls", "2stage-CPLS")]:
    d = imp[dataset].dropna(subset=["LSTM (m)"]).rename_axis("vessel type").reset_index()
    bars.append(d.hvplot.bar(x="vessel type", y=["curvilinear (m)", "LSTM (m)", "CLSTM (m)"], rot=60, width=620, height=380,
                             color=[GREY, BLUE, ORANGE], ylabel="metres", logy=True,
                             title=f"Mean point-wise error on the test trips, {title} data (log scale)"))
hv.Layout(bars).cols(1)

:Layout
   .Bars.I  :Bars   [vessel type,Variable]   (value)
   .Bars.II :Bars   [vessel type,Variable]   (value)

### Each model on the other dataset's test samples

A model trained on one dataset can also be asked to predict the other's test samples (the same test trips, but the other dataset's origins and `dt`). This is computable, because the
network only sees `(x', y', tau)`, but it is an **out-of-distribution test, not a like-for-like one**: the two datasets have very different `dt`, hence very different `tau`
(Eq. 11 of the paper makes the offset the input), and a network trained on the clean samples has never seen offsets as large as those of the compressed samples, and vice versa.
The numbers answer a practical question, "what happens if the model is used on data of the other kind", not "which training is better". The `all` rows (mean point-wise error, metres):

In [14]:
cross, cross_tables = [], {}
for trained in ("uncompressed", "2stage_cpls"):
    for evaluated in ("uncompressed", "2stage_cpls"):
        test = TEST[evaluated]
        for kind in ("lstm", "clstm"):
            if trained == evaluated:
                table = tables[trained, kind]
            else:
                table = cross_tables[trained, evaluated, kind] = prediction_errors(
                    models[f"{kind}_{trained}"].predict(test), test)
            cross.append({"model": kind.upper(), "trained on": trained, "tested on": evaluated,
                          "samples scored": int(table.loc["all", "samples"]), "mean_error (m)": table.loc["all", "mean_error"],
                          "median_error (m)": table.loc["all", "median_error"], "rmse (m)": table.loc["all", "rmse"]})
        cross.append({"model": "curvilinear", "trained on": "-", "tested on": evaluated,
                      "samples scored": int(tables[evaluated, "curvilinear"].loc["all", "samples"]),
                      "mean_error (m)": tables[evaluated, "curvilinear"].loc["all", "mean_error"],
                      "median_error (m)": tables[evaluated, "curvilinear"].loc["all", "median_error"],
                      "rmse (m)": tables[evaluated, "curvilinear"].loc["all", "rmse"]})
cross = pd.DataFrame(cross).drop_duplicates(["model", "trained on", "tested on"]).reset_index(drop=True)
cross.round(2)

,model,trained on,tested on,samples scored,mean_error (m),median_error (m),rmse (m)
0,LSTM,uncompressed,uncompressed,2690936,25.26,8.99,84.46
1,CLSTM,uncompressed,uncompressed,2690936,21.16,5.57,81.91
2,curvilinear,-,uncompressed,2690936,17.61,2.71,85.00
3,LSTM,uncompressed,2stage_cpls,294940,803.66,94.18,3967.06
4,CLSTM,uncompressed,2stage_cpls,294940,823.80,94.40,4033.73
5,curvilinear,-,2stage_cpls,294940,811.88,72.44,4045.40
6,LSTM,2stage_cpls,uncompressed,2690936,197.08,135.59,271.72
7,CLSTM,2stage_cpls,uncompressed,2690936,273.23,143.78,613.78
8,LSTM,2stage_cpls,2stage_cpls,294940,564.00,187.35,1565.02
9,CLSTM,2stage_cpls,2stage_cpls,294940,603.07,150.79,2588.61


### Supplementary: samples without an internal gap

**Supplementary, not the method's evaluation.** A trip can contain a long silent interval, and a sample whose prediction window `[t, t + 3 dt]` spans one has a target that is only an interpolation across the gap (C10 keeps it,
as the method is defined). Here the test samples are re-scored without those whose window overlaps an interval between two consecutive vertices **longer than 10 `dt`** (`dt` of the trip, Eq. 12, of the dataset in question).
The models are the same ones; only the scored samples change, and the number excluded is printed.

In [15]:
def gap_free(samples, points, factor=10.0):
    # True where no interval between consecutive vertices overlapping [t, t + 3 dt] exceeds factor * dt
    times = {k: np.unique(v.to_numpy()) for k, v in points.groupby("trip_id").unix_time}   # sorted, repeated times once
    keep = np.ones(len(samples), bool)
    codes, uniques = pd.factorize(pd.Series(samples.trajectory))
    order = np.argsort(codes, kind="stable")
    bounds = np.flatnonzero(np.diff(codes[order])) + 1
    for part in np.split(order, bounds):
        t = times[uniques[codes[part[0]]]]
        dt = samples.dt[part[0]]
        cs = np.concatenate([[0], np.cumsum(np.diff(t) > factor * dt)])
        s0, e0 = samples.time[part], samples.time[part] + 3 * dt
        i0 = np.clip(np.searchsorted(t, s0, side="right") - 1, 0, len(t) - 2)
        i1 = np.clip(np.searchsorted(t, e0, side="left") - 1, 0, len(t) - 2)
        keep[part] = cs[i1 + 1] - cs[i0] == 0
    return keep

tables_gf, imp_gf, gf_counts = {}, {}, {}
for dataset, points in [("uncompressed", raw), ("2stage_cpls", cpls)]:
    test = TEST[dataset]
    mask = gap_free(test, points)
    sub = test.subset(mask)
    gf_counts[dataset] = (len(test), int(mask.sum()))
    print(f"{dataset:13s} {len(test) - mask.sum():,} of {len(test):,} test samples excluded ({100 * (1 - mask.mean()):.2f} %), {mask.sum():,} kept")
    tables_gf[dataset, "curvilinear"] = prediction_errors(sub.curvilinear, sub)
    for kind in ("lstm", "clstm"):
        tables_gf[dataset, kind] = prediction_errors(preds[dataset, kind][mask], sub)
    ok = ~np.isnan(preds[dataset, "clstm"][mask]).any(axis=(1, 2))
    both = sub.subset(ok)
    for kind, p in [("curvilinear", both.curvilinear), ("lstm", preds[dataset, "lstm"][mask][ok]), ("clstm", preds[dataset, "clstm"][mask][ok])]:
        tables_gf[dataset, kind + "_same"] = prediction_errors(p, both)
    imp_gf[dataset] = improvement(dataset, tables_gf)

uncompressed  33,386 of 2,690,936 test samples excluded (1.24 %), 2,657,550 kept


2stage_cpls   25,419 of 294,940 test samples excluded (8.62 %), 269,521 kept


In [16]:
for dataset in ("uncompressed", "2stage_cpls"):
    print(f"=== {dataset}: supplementary, test samples without an internal gap > 10 dt ===")
    for kind in ("curvilinear", "lstm", "clstm"):
        print(f"--- {kind} ---")
        print(tables_gf[dataset, kind].round(2).to_string())
    print("--- mean point-wise error and improvements, on the samples the CLSTM predicts ---")
    print(imp_gf[dataset].round(2).to_string())
    print()

=== uncompressed: supplementary, test samples without an internal gap > 10 dt ===
--- curvilinear ---
                  samples  missing    rmse  rmse_t+1  rmse_t+2  rmse_t+3  mean_error  median_error
Anti-Pollution      21717        0  165.39     49.79    141.58    244.00       34.06          5.77
Cargo              242961        0   49.18     21.29     41.35     71.37       18.85          7.11
Dive Vessel           509        0  390.02    155.94    351.39    555.48      120.95         35.14
Dredger             31712        0   99.32     33.50     84.49    146.06       31.92         15.74
Fishing            526617        0   87.32     35.11     77.16    125.26       25.08          6.68
High-Speed Craft        8        0  126.79     63.87    115.75    175.36      113.99        108.07
Law Enforce          3025        0  127.46     79.70    126.83    162.18       14.92          7.00
Local Vessel        27278        0    5.72      3.95      5.76      7.03        3.20          1.62
Militar

## 6 · An example, then save

One test trip: the actual track, and for a few consecutive origins the three positions of the curvilinear model, the LSTM and the CLSTM against the actual ones. The
origins are those around the sample where the curvilinear model is the furthest off, so this shows the largest correction the networks have a chance to make (clean data).

In [17]:
d = "uncompressed"
traj = pd.Series(test_raw.trajectory)
counts = traj.value_counts()
cand = counts[counts >= 50]
trip = (cand - 500).abs().sort_values().index[0] if len(cand) else counts.index[0]
idx = np.flatnonzero(traj.to_numpy() == trip)
track = raw[raw.trip_id == trip]
cur_err = np.hypot(*(test_raw.curvilinear[idx] - test_raw.targets[idx]).transpose(2, 0, 1)).mean(axis=1)
worst = int(np.argmax(cur_err))
pick = idx[max(worst - 2, 0):worst + 4]

def merc(a):
    a = np.asarray(a, dtype=float).reshape(-1, 2)
    mx, my = TO_MERCATOR(a[:, 0], a[:, 1])
    return np.column_stack([mx, my])

layers = [hv.Tiles("https://basemaps.cartocdn.com/light_all/{Z}/{X}/{Y}.png").opts(alpha=0.8),
          hv.Path([merc(track[["x", "y"]].to_numpy())]).opts(color="#999999", line_width=1.5)]
for name, color, source in [("actual", "black", lambda i: test_raw.targets[i]), ("curvilinear", GREY, lambda i: test_raw.curvilinear[i]),
                            ("LSTM", BLUE, lambda i: preds[d, "lstm"][i]), ("CLSTM", ORANGE, lambda i: preds[d, "clstm"][i])]:
    lines = [merc(np.vstack([test_raw.origin[i], source(i)])) for i in pick]
    layers.append(hv.Path(lines, label=name).opts(color=color, line_width=2.5 if name == "actual" else 1.5))
    layers.append(hv.Points(np.vstack(lines), label=name).opts(color=color, size=5))
box = merc(np.vstack([test_raw.origin[pick], test_raw.targets[pick].reshape(-1, 2), test_raw.curvilinear[pick].reshape(-1, 2)]))
pad = 0.3 * max(np.ptp(box[:, 0]), np.ptp(box[:, 1]), 200)
hv.Overlay(layers).opts(width=720, height=620, xaxis=None, yaxis=None, active_tools=["wheel_zoom"], show_legend=True, legend_position="right",
                        xlim=(box[:, 0].min() - pad, box[:, 0].max() + pad), ylim=(box[:, 1].min() - pad, box[:, 1].max() + pad),
                        title=f"Test trip {trip} ({test_raw.group[idx[0]]}), {len(pick)} origins: actual, curvilinear, LSTM and CLSTM")

:Overlay
   .Tiles.I            :Tiles   [x,y]
   .Path.I             :Path   [x,y]
   .Path.Actual        :Path   [x,y]
   .Points.Actual      :Points   [x,y]
   .Path.Curvilinear   :Path   [x,y]
   .Points.Curvilinear :Points   [x,y]
   .Path.LSTM          :Path   [x,y]
   .Points.LSTM        :Points   [x,y]
   .Path.CLSTM         :Path   [x,y]
   .Points.CLSTM       :Points   [x,y]

**Save.** Written to `results/05_clstm_prediction/<DATA>/`: `errors.csv` (every error table, with the dataset, the model and what it was tested on), `summary.txt`
(the `summary()` of each model, the counts at every stage, the timings, the device) and the four models (`<model>.pt`, written by `ClstmPredictor.save`, readable with
`ClstmPredictor.load` and `weights_only=True`). A saved model is read back and its predictions are checked against the original.

In [18]:
rows = []
for (dataset, kind), table in tables.items():
    if kind.endswith("_same"):
        continue
    t = table.copy()
    t.insert(0, "group", t.index)
    t.insert(0, "tested_on", dataset)
    t.insert(0, "trained_on", "-" if kind == "curvilinear" else dataset)
    t.insert(0, "model", kind)
    t.insert(0, "subset", "all test samples")
    rows.append(t)
for (trained, evaluated, kind), t in cross_tables.items():
    t = t.copy()
    t.insert(0, "group", t.index)
    t.insert(0, "tested_on", evaluated)
    t.insert(0, "trained_on", trained)
    t.insert(0, "model", kind)
    t.insert(0, "subset", "all test samples")
    rows.append(t)
for (dataset, kind), table in tables_gf.items():
    if kind.endswith("_same"):
        continue
    t = table.copy()
    t.insert(0, "group", t.index)
    t.insert(0, "tested_on", dataset)
    t.insert(0, "trained_on", "-" if kind == "curvilinear" else dataset)
    t.insert(0, "model", kind)
    t.insert(0, "subset", "supplementary: no gap > 10 dt in the window")
    rows.append(t)
errors = pd.concat(rows, ignore_index=True)
errors.to_csv(OUT / "errors.csv", index=False)

for name, model in models.items():
    model.save(OUT / f"{name}.pt")
back = ClstmPredictor.load(OUT / "clstm_uncompressed.pt", device=DEVICE)
same = np.allclose(back.predict(test_raw), preds["uncompressed", "clstm"], equal_nan=True, atol=1e-3)
print("saved model reloads with the same predictions (to 1 mm):", same)

lines = [f"CLSTM prediction, DATA = {DATA}", f"device {resolved}, GPU {GPU_NAME}, torch {torch.__version__}", "",
         "== counts ==",
         summary_in.to_string(), "",
         f"samples (clean): {len(samples_raw):,} ({len(train_raw):,} train, {len(test_raw):,} test)",
         f"samples (2stage-CPLS): {len(samples_cpls):,} ({len(train_cpls):,} train, {len(test_cpls):,} test)",
         f"trips in the split: {len(train_ids):,} train, {len(test_ids):,} test (test_size={TEST_SIZE}, seed={SEED}; same ids for both)", "",
         "== timings (s) ==", f"make_samples clean: {SAMPLES_RAW_S:.1f}", f"make_samples 2stage-CPLS: {SAMPLES_CPLS_S:.1f}"]
lines += [f"fit {name}: {s:.1f}" for name, s in fit_seconds.items()]
lines += ["", "== CLSTM vs LSTM, mean point-wise error, same scored samples =="]
for d in ("uncompressed", "2stage_cpls"):
    lines += [f"-- {d} --", imp[d].round(2).to_string()]
lines += ["", "== supplementary: test samples without a gap > 10 dt in the window =="]
for d in ("uncompressed", "2stage_cpls"):
    lines += [f"-- {d}: {gf_counts[d][0] - gf_counts[d][1]:,} of {gf_counts[d][0]:,} test samples excluded --", imp_gf[d].round(2).to_string()]
lines += ["", "== each model on the other dataset =="] + [cross.round(2).to_string()]
for name, model in models.items():
    lines += ["", f"== {name} ==", model.summary()]
(OUT / "summary.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
for f in sorted(OUT.iterdir()):
    print(f"{f.name:26s} {f.stat().st_size / 1e6:8.2f} MB")

saved model reloads with the same predictions (to 1 mm): True
clstm_2stage_cpls.pt           1.29 MB
clstm_uncompressed.pt          1.28 MB
errors.csv                     0.05 MB
lstm_2stage_cpls.pt            0.08 MB
lstm_uncompressed.pt           0.08 MB
progress.log                   0.00 MB
summary.txt                    0.02 MB


### Results in numbers

In [19]:
for d, title in [("uncompressed", "clean"), ("2stage_cpls", "2stage-CPLS")]:
    r = imp[d].loc["all"]
    print(f"{title:12s} {int(r['samples']):>9,} scored test samples   mean error: curvilinear {r['curvilinear (m)']:.1f} m, "
          f"LSTM {r['LSTM (m)']:.1f} m, CLSTM {r['CLSTM (m)']:.1f} m   "
          f"CLSTM vs LSTM {r['CLSTM vs LSTM (%)']:+.2f} %   LSTM vs curvilinear {r['LSTM vs curvilinear (%)']:+.2f} %")
print(f"\ntotal training time {sum(fit_seconds.values()):.0f} s on {resolved} ({GPU_NAME}); DATA = {DATA}")

clean        2,690,936 scored test samples   mean error: curvilinear 17.6 m, LSTM 25.3 m, CLSTM 21.2 m   CLSTM vs LSTM +16.23 %   LSTM vs curvilinear -43.46 %
2stage-CPLS    294,940 scored test samples   mean error: curvilinear 811.9 m, LSTM 564.0 m, CLSTM 603.1 m   CLSTM vs LSTM -6.93 %   LSTM vs curvilinear +30.53 %

total training time 2889 s on cuda (NVIDIA GeForce RTX 4060 Ti); DATA = full


## 7 · Notes: what the comparison means, and what it does not

**What was done.** The same networks (H1-H6, library defaults) were trained on two sets of samples built from the same trips and evaluated on the same held-out
trips: 80 % of the trips for training (10 % of those for validation), 20 % for testing.

**What the two datasets are.** The prediction target is a position `dt`, `2 dt`, `3 dt` ahead, with `dt` the trip's mean interval between its vertices (Eq. 12).
* On the **clean** trips `dt` is the raw sampling interval and every AIS point is an origin: short horizons, millions of samples on the full archive.
* On the **compressed** trips `dt` is the mean interval between *kept* vertices, much larger (the `median dt` is printed in section 2), only the kept vertices are origins, and the target
  is interpolated *between kept vertices* (a straight line, as the compression assumes). Horizons are longer, the speed and course at an origin are those of a vertex chosen
  because the vessel was turning or changing speed, and there are an order of magnitude fewer samples.

**What the comparison does mean.** For each dataset separately, the table gives the curvilinear baseline, the LSTM and the CLSTM, so the questions "does a learned correction help on top of dead
reckoning?" and "does a network per vessel type help on top of a single one?" are answered once on clean data and once on compressed data, with the same split. The
relative improvement of CLSTM over LSTM is a ratio within a dataset, so it can be put side by side across datasets; the absolute errors cannot.

**What it does not mean.** Absolute errors of the two datasets are not comparable: they predict different horizons from different origins, and the compressed targets are
themselves an approximation of the real track (a long straight interpolation), so a small error against them is not a small error against the vessel. A lower error on the
compressed data therefore does **not** show that compression helps prediction, and a higher one does not show it hurts; what the experiment shows is how each training
behaves on its own kind of data, and, in the cross-evaluation, how a model behaves when `dt` differs from its training data. Neither is it a measurement of how much information
compression loses.
Within the 2stage-CPLS data the split by trip still holds, but a compressed trip has few vertices, so the validation and test sets are small and noisy, especially per vessel type.

**On the data and the paper.** The 15.68 % reported by Mehri et al. (2021) is on their data, period and vessel types; nothing here reproduces or confirms it.
`Unknown` (no usable static type in the preprocessed data) was mapped to `Other`, the paper's group of vessels whose type is null.
Types with fewer than 200 training samples have no network and are listed under `dropped (fewer than 200 samples)` in each summary; their test samples are not scored by the CLSTM,
which is why the table of improvements compares the LSTM on exactly the samples the CLSTM scored.

**The sample is too small for meaningful accuracy.** Two days, 565 vessels, a few hundred trips and few vessels per type: the networks of the small types train on a handful of
trips, the split leaves a few dozen test trips, and anything that differs between LSTM and CLSTM here is within what a different seed or split would change. Read the sample run as a check that
the pipeline works, not as a result. The numbers worth reading come from the full-archive run (`notebooks/full_runs/05_clstm_prediction.ipynb`).

**What the network learns.** The deviation `actual - T'` (Eq. 2), added back to the curvilinear positions; see the section before the evaluation for the diagnostic that led to it. The outputs
are absolute positions, so the baseline and the networks are scored on the same footing. A vessel type whose few training trips cover only part of the area is still a weak network: it has little to learn from.

**Gaps.** A sample's target is interpolated linearly in time along the trip (C10), and a trip can contain a long silent interval (a vessel out of AIS coverage and back). A target interpolated across such a
gap is not where the vessel was, and a few of these dominate the RMSE (root of the mean of squares) while barely moving the mean or median. The main tables keep every sample (as the method is defined);
the supplementary tables repeat the evaluation without the samples whose three-step window contains a gap longer than 10 `dt`.

**Settings, in plain language** (H1-H6 and C-decisions of `CLSTM_GUIDE.md`): constant speed and course for the curvilinear model; speed and course are the AIS SOG and COG of the origin row; unavailable values drop the
sample; grid north is taken as north; a trajectory is a trip; the split is by trip with a fixed seed; the networks are initialised from the same seed so that one group's network does not depend on the others.